**Lab type:** review  
**Course:** DS201 — Data Wrangling & Feature Engineering  
**Lesson:** Encoding Categorical Features  
**Task:** A colleague's encoding pipeline is correct and runs without errors. Your job is to evaluate the encoding choices and judge whether they are appropriate for the data and the downstream model.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder

np.random.seed(42)

# Synthetic dataset
n = 500
df = pd.DataFrame({
    'plan': np.random.choice(['starter', 'pro', 'enterprise', 'unknown'], n),
    'region': np.random.choice(['APAC', 'EMEA', 'LATAM', 'NA'], n),
    'industry': np.random.choice(['SaaS', 'FinTech', 'Retail', 'Healthcare', 'Manufacturing', 'Education',
                                    'Media', 'Telecom', 'Energy', 'Travel', 'Real Estate', 'Other'], n),
    'churned': np.random.randint(0, 2, n),
})

print(df.head())

## The encoding pipeline to review

In [ ]:
# COLLEAGUE'S PIPELINE

# 1. Ordinal encoding for plan
plan_order = {'starter': 0, 'pro': 1, 'enterprise': 2, 'unknown': -1}
df['plan_encoded'] = df['plan'].map(plan_order)

# 2. One-hot encoding for region using pd.get_dummies
df_encoded = pd.get_dummies(df, columns=['region'], drop_first=True)

# 3. OrdinalEncoder for industry (no explicit categories)
enc_industry = OrdinalEncoder()
df_encoded['industry_encoded'] = enc_industry.fit_transform(df[['industry']])

# 4. Frequency encoding for high-cardinality backup
freq_map = df['industry'].value_counts(normalize=True)
df_encoded['industry_freq'] = df['industry'].map(freq_map)

print("Pipeline complete. Encoded columns:")
print([c for c in df_encoded.columns if 'encoded' in c or 'region' in c or 'freq' in c])

## Review Question 1: Ordinal encoding for `plan`

**Question:** The code uses a hand-coded dict for ordinal encoding of `plan`. When would you prefer this over `sklearn.OrdinalEncoder`? When would you use `sklearn` instead?

In [ ]:
# Your analysis here
print("Hand-coded dict advantages:")
print("- Explicit control over ordering")
print("- Handles out-of-order categories (e.g., 'unknown' → -1)")
print("- Clear intent in the code")
print("\nWhen to use sklearn.OrdinalEncoder:")
print("- More features to encode (less boilerplate)")
print("- Consistent treatment across many columns")
print("- Pipeline integration")

## Review Question 2: One-hot encoding with `pd.get_dummies`

**Question:** Why is `pd.get_dummies` unsafe for production? How would you fix it?

In [ ]:
# Test: what happens with a new region at prediction time?
new_customer = pd.DataFrame([{'plan': 'pro', 'region': 'MENA', 'industry': 'SaaS', 'churned': 0}])

# get_dummies on new data produces different columns
new_encoded = pd.get_dummies(new_customer, columns=['region'])
print(f"Training columns: {sorted([c for c in df_encoded.columns if 'region' in c])}")
print(f"New data columns:  {sorted([c for c in new_encoded.columns if 'region' in c])}")
print(f"\nMismatch: 'MENA' creates a new column not in training.")
print(f"\nFix: Use sklearn.OneHotEncoder(handle_unknown='ignore')")

## Review Question 3: OrdinalEncoder without explicit categories

**Question:** The OrdinalEncoder is applied to `industry` with no explicit `categories` parameter. What does this mean, and is it a problem?

In [ ]:
# Check the category ordering
category_mapping = dict(zip(enc_industry.categories_[0], range(len(enc_industry.categories_[0]))))
print("OrdinalEncoder category → number mapping:")
for cat, num in sorted(category_mapping.items(), key=lambda x: x[1]):
    print(f"  {cat}: {num}")

print("\nAnalysis:")
print("- Categories are alphabetical (default sklearn behavior)")
print("- Industry has no real ordering (SaaS is not 'less than' Telecom)")
print("- A linear model will infer a meaningless coefficient")
print("- Better: use one-hot encoding OR hand-code if there's domain ordering")

## Review Question 4: Handling unseen categories

**Question:** A new customer arrives with `region='MENA'` at prediction time. Trace through what happens with `pd.get_dummies` vs `OneHotEncoder(handle_unknown='ignore')`.

In [ ]:
# pd.get_dummies: crashes or misaligns
print("pd.get_dummies behavior:")
print("- Creates a 'region_MENA' column (new)")
print("- Doesn't create 'region_APAC', 'region_EMEA' (missing)")
print("- Shape mismatch: model expects 2 region columns, gets 1 (new)")
print("- Result: crash or silent misalignment")

print("\nOneHotEncoder(handle_unknown='ignore') behavior:")
print("- 'MENA' is unseen")
print("- Returns [0, 0] for all known region columns")
print("- Shape matches: exactly 4 columns expected")
print("- Result: graceful handling, no crash")

## Review Question 5: Evaluating the pipeline

**Question:** An AI tool generated this entire encoding pipeline in one shot. What are the top 3 checks you would run before using it in production?

In [ ]:
print("Top 3 checks for AI-generated encoding code:")
print("\n1. Verify ordinal/one-hot choice matches feature type")
print("   - 'plan' has ordering (starter → pro → enterprise) → ordinal is correct")
print("   - 'region' has no ordering → one-hot is correct (but use sklearn, not pd.get_dummies)")
print("   - 'industry' has no ordering → one-hot is correct (current code uses ordinal with alphabet ordering)")

print("\n2. Check how unseen categories are handled")
print("   - pd.get_dummies: NOT safe for production")
print("   - OrdinalEncoder without handle_unknown: crashes on unseen")
print("   - Solution: use OneHotEncoder(handle_unknown='ignore')")

print("\n3. Test on a small new batch that includes unseen values")
print("   - Pass data with a new region/industry")
print("   - Verify no crash and correct output shape")
print("   - Confirm model can predict without error")